# MOSHIKO 2.0 — Tests
Prerequisite: Run moshiko2.0.ipynb first. Models (mimi, moshi_lm, lm_gen) must be loaded.

Speech-to-speech (Test 3) is in separate notebooks:
- test3_OptionB.ipynb: LMGen with float32 fix
- test3_OptionC.ipynb: InferenceState official API (recommended)

In [ ]:
# Test 1: Mimi encode/decode with synthetic audio
import torch, soundfile as sf
import numpy as np, time
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000
DURATION = 3

# Create 440Hz tone
t = torch.linspace(0, DURATION, SAMPLE_RATE * DURATION)
wav = (0.3 * torch.sin(2 * np.pi * 440 * t)).unsqueeze(0).unsqueeze(0)
print(f"Input shape: {wav.shape} | Sample rate: {SAMPLE_RATE} Hz")

# Save input
input_path = f"{FOLDERS['audio_in']}/test1_input_440hz.wav"
sf.write(input_path, wav.squeeze().numpy(), SAMPLE_RATE)

# Encode
encode_start = time.time()
with torch.no_grad():
    codes = mimi.encode(wav.to(DEVICE))
encode_time = time.time() - encode_start
print(f"Encoded shape: {codes.shape} | Encode time: {encode_time*1000:.1f} ms")

# Decode
decode_start = time.time()
with torch.no_grad():
    decoded = mimi.decode(codes)
decode_time = time.time() - decode_start
print(f"Output shape: {decoded.shape} | Decode time: {decode_time*1000:.1f} ms")

# Save output
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f"{FOLDERS['audio_out']}/test1_output_{ts}.wav"
sf.write(output_path, decoded.squeeze().cpu().numpy(), SAMPLE_RATE)

# SNR calculation
min_len = min(wav.squeeze().shape[0], decoded.squeeze().cpu().shape[0])
original = wav.squeeze().numpy()[:min_len]
reconstructed = decoded.squeeze().cpu().numpy()[:min_len]
noise = original - reconstructed
snr = 10 * np.log10(np.mean(original**2) / (np.mean(noise**2) + 1e-8))
print(f"SNR: {snr:.2f} dB")

# Play
print("\nInput audio:")
display(Audio(original, rate=SAMPLE_RATE))
print("Output audio (after encode/decode):")
display(Audio(reconstructed, rate=SAMPLE_RATE))

# Save results
result = {
    "test": "Mimi encode/decode (synthetic 440Hz)",
    "timestamp": ts,
    "duration_s": DURATION,
    "encode_ms": round(encode_time * 1000, 2),
    "decode_ms": round(decode_time * 1000, 2),
    "snr_db": round(float(snr), 2),
    "input_file": input_path,
    "output_file": output_path,
}
import json
with open(f"{FOLDERS['outputs']}/test1_results_{ts}.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"\nTest 1 complete. SNR={snr:.2f}dB. Files saved to Drive.")

In [ ]:
# Test 2: Mimi encode/decode with real audio file
# Upload a WAV file to Colab first, or provide a path
import torch, soundfile as sf
import numpy as np, time, os
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000

# Option A: Use a file from audio_input folder
# Option B: Upload via Colab files panel
# Replace this with your actual file path
audio_file = f"{FOLDERS['audio_in']}/test_input_440hz.wav"  # default: use test1 output

if not os.path.exists(audio_file):
    print(f"File not found: {audio_file}")
    print("Upload a WAV file to Colab and set audio_file path above.")
    print("File must be: WAV format, 24kHz, mono.")
else:
    # Load audio
    wav_np, sr = sf.read(audio_file)
    if sr != SAMPLE_RATE:
        print(f"Warning: Sample rate {sr} != {SAMPLE_RATE}. Resampling.")
        import torchaudio.transforms as T
        wav_tensor = torch.from_numpy(wav_np).float().unsqueeze(0)
        resampler = T.Resample(sr, SAMPLE_RATE)
        wav_tensor = resampler(wav_tensor)
        wav_np = wav_tensor.squeeze().numpy()

    wav = torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0)
    print(f"Loaded: {audio_file}")
    print(f"Duration: {wav.shape[-1]/SAMPLE_RATE:.2f}s | Shape: {wav.shape}")

    # Encode
    encode_start = time.time()
    with torch.no_grad():
        codes = mimi.encode(wav.to(DEVICE))
    encode_time = time.time() - encode_start
    print(f"Encoded shape: {codes.shape} | Encode time: {encode_time*1000:.1f} ms")

    # Decode
    decode_start = time.time()
    with torch.no_grad():
        decoded = mimi.decode(codes)
    decode_time = time.time() - decode_start
    print(f"Output shape: {decoded.shape} | Decode time: {decode_time*1000:.1f} ms")

    # Save output
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = f"{FOLDERS['audio_out']}/test2_output_{ts}.wav"
    sf.write(output_path, decoded.squeeze().cpu().numpy(), SAMPLE_RATE)

    # SNR
    min_len = min(wav.squeeze().shape[0], decoded.squeeze().cpu().shape[0])
    original = wav.squeeze().numpy()[:min_len]
    reconstructed = decoded.squeeze().cpu().numpy()[:min_len]
    noise = original - reconstructed
    snr = 10 * np.log10(np.mean(original**2) / (np.mean(noise**2) + 1e-8))
    print(f"SNR: {snr:.2f} dB")

    # Play
    print("\nInput audio:")
    display(Audio(original, rate=SAMPLE_RATE))
    print("Output audio (after encode/decode):")
    display(Audio(reconstructed, rate=SAMPLE_RATE))

    # Save results
    result = {
        "test": "Mimi encode/decode (real audio file)",
        "timestamp": ts,
        "input_file": audio_file,
        "output_file": output_path,
        "duration_s": round(wav.shape[-1]/SAMPLE_RATE, 2),
        "encode_ms": round(encode_time * 1000, 2),
        "decode_ms": round(decode_time * 1000, 2),
        "snr_db": round(float(snr), 2),
    }
    import json
    with open(f"{FOLDERS['outputs']}/test2_results_{ts}.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nTest 2 complete. SNR={snr:.2f}dB. Files saved to Drive.")

In [ ]:
# Test 3: Speech-to-speech — moved to separate notebooks
# See test3_OptionB.ipynb (LMGen with float32 fix) or
# test3_OptionC.ipynb (InferenceState official API)
print("Test 3 moved to separate notebooks.")
print("Run test3_OptionB.ipynb or test3_OptionC.ipynb for speech-to-speech.")

In [ ]:
# Test 4: Latency benchmark
import torch, numpy as np, json, time
import matplotlib.pyplot as plt
from datetime import datetime

SAMPLE_RATE = 24000
WARMUP_RUNS = 3
BENCH_RUNS = 5
DURATIONS = [1, 3, 5, 10]

print("Starting latency benchmark...")
print(f"  Warmup: {WARMUP_RUNS} runs | Benchmark: {BENCH_RUNS} runs each\n")

results = []

for dur in DURATIONS:
    print(f"  Testing {dur}s audio...", end="", flush=True)

    wav = (0.3 * torch.sin(2 * np.pi * 440 *
           torch.linspace(0, dur, SAMPLE_RATE * dur))
          ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Warmup
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            codes = mimi.encode(wav)
            _ = mimi.decode(codes)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    # Benchmark encode
    enc_times = []
    for _ in range(BENCH_RUNS):
        if DEVICE == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            codes = mimi.encode(wav)
        if DEVICE == "cuda": torch.cuda.synchronize()
        enc_times.append(time.perf_counter() - t0)

    # Benchmark decode
    dec_times = []
    for _ in range(BENCH_RUNS):
        if DEVICE == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = mimi.decode(codes)
        if DEVICE == "cuda": torch.cuda.synchronize()
        dec_times.append(time.perf_counter() - t0)

    enc_mean = np.mean(enc_times) * 1000
    dec_mean = np.mean(dec_times) * 1000
    total_ms = enc_mean + dec_mean
    rtf = total_ms / (dur * 1000)
    mem_mb = torch.cuda.memory_allocated() / 1e6 if DEVICE == "cuda" else 0

    results.append({
        "duration_s": dur,
        "encode_ms": round(enc_mean, 2),
        "decode_ms": round(dec_mean, 2),
        "total_ms": round(total_ms, 2),
        "rtf": round(rtf, 4),
        "gpu_mem_mb": round(mem_mb, 1),
    })
    print(f" Encode:{enc_mean:.0f}ms | Decode:{dec_mean:.0f}ms | RTF:{rtf:.3f}")

# Print table
print("\nBENCHMARK RESULTS")
print(f"  {'Duration':>8} {'Encode(ms)':>12} {'Decode(ms)':>12} {'Total(ms)':>11} {'RTF':>8}")
print("  " + "-"*55)
for r in results:
    status = "Faster than RT" if r["rtf"] < 1.0 else "Slower than RT"
    print(f"  {r['duration_s']:>7}s {r['encode_ms']:>12.1f} "
          f"{r['decode_ms']:>12.1f} {r['total_ms']:>11.1f} "
          f"{r['rtf']:>8.3f} ({status})")
print("\nNote: RTF < 1.0 = faster than real-time")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Moshiko Latency Benchmark", fontsize=13, fontweight="bold")

durs = [r["duration_s"] for r in results]
encs = [r["encode_ms"] for r in results]
decs = [r["decode_ms"] for r in results]
rtfs = [r["rtf"] for r in results]

x = np.arange(len(durs))
w = 0.35
axes[0].bar(x - w/2, encs, w, label="Encode", color="#2196F3")
axes[0].bar(x + w/2, decs, w, label="Decode", color="#FF9800")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{d}s" for d in durs])
axes[0].set_xlabel("Audio Duration")
axes[0].set_ylabel("Time (ms)")
axes[0].set_title("Encode vs Decode Latency")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

colors = ["#4CAF50" if r < 1.0 else "#F44336" for r in rtfs]
axes[1].bar([f"{d}s" for d in durs], rtfs, color=colors)
axes[1].axhline(1.0, color="red", linestyle="--", label="Real-time threshold")
axes[1].set_xlabel("Audio Duration")
axes[1].set_ylabel("Real-Time Factor (RTF)")
axes[1].set_title("Real-Time Factor (lower = faster)")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = f"{FOLDERS['benchmarks']}/latency_{ts}.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

json_path = f"{FOLDERS['benchmarks']}/latency_{ts}.json"
with open(json_path, "w") as f:
    json.dump({"benchmark": results, "device": DEVICE}, f, indent=2)

print(f"\nTest 4 complete. Plot and data saved to Drive.")

In [ ]:
# Test 5: BF16 vs Q8 quality comparison
import torch, numpy as np, soundfile as sf, json
import matplotlib.pyplot as plt, time
from IPython.display import Audio, display
from datetime import datetime
from moshi.models import loaders

SAMPLE_RATE = 24000

# Check bf16 Mimi exists
mimi_bf16_path = f"{FOLDERS['models_bf16']}/tokenizer-e351c8d8-checkpoint125.safetensors"
if not os.path.exists(mimi_bf16_path):
    print("BF16 Mimi not found. Download it in moshiko2.0.ipynb first.")
    raise FileNotFoundError(mimi_bf16_path)

# Load BF16 Mimi
print("Loading BF16 Mimi...")
mimi_bf16 = loaders.get_mimi(mimi_bf16_path, device=DEVICE)
mimi_bf16.set_num_codebooks(8)
mimi_bf16.eval()
print("  BF16 Mimi loaded")

# Test signals
test_signals = {
    "Pure tone 440Hz": lambda: 0.3 * torch.sin(2 * np.pi * 440 *
                               torch.linspace(0, 2, SAMPLE_RATE * 2)),
    "Mixed tones": lambda: 0.2 * (
                               torch.sin(2 * np.pi * 440 * torch.linspace(0, 2, SAMPLE_RATE*2)) +
                               torch.sin(2 * np.pi * 880 * torch.linspace(0, 2, SAMPLE_RATE*2)) +
                               torch.sin(2 * np.pi * 220 * torch.linspace(0, 2, SAMPLE_RATE*2))),
    "White noise": lambda: 0.1 * torch.randn(SAMPLE_RATE * 2),
    "Chirp (sweep)": lambda: 0.3 * torch.sin(2 * np.pi *
                               (200 + 1800 * torch.linspace(0, 1, SAMPLE_RATE*2)**2) *
                               torch.linspace(0, 2, SAMPLE_RATE*2)),
}

comparison_results = []

print("\nRunning comparison...\n")
print(f"  {'Signal':<20} {'Q8 SNR':>10} {'BF16 SNR':>10} {'Diff':>8} {'Q8 faster?':>12}")
print("  " + "-"*65)

for signal_name, signal_fn in test_signals.items():
    wav = signal_fn().unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Q8 encode-decode
    t0 = time.perf_counter()
    with torch.no_grad():
        codes_q8 = mimi.encode(wav)
        out_q8 = mimi.decode(codes_q8)
    time_q8 = (time.perf_counter() - t0) * 1000

    # BF16 encode-decode
    t0 = time.perf_counter()
    with torch.no_grad():
        codes_bf16 = mimi_bf16.encode(wav)
        out_bf16 = mimi_bf16.decode(codes_bf16)
    time_bf16 = (time.perf_counter() - t0) * 1000

    # SNR
    def compute_snr(original, reconstructed):
        min_len = min(original.shape[-1], reconstructed.shape[-1])
        orig = original.squeeze().cpu().numpy()[:min_len].astype(np.float32)
        rec = reconstructed.squeeze().cpu().numpy()[:min_len].astype(np.float32)
        noise = orig - rec
        return 10 * np.log10(np.mean(orig**2) / (np.mean(noise**2) + 1e-8))

    snr_q8 = compute_snr(wav, out_q8)
    snr_bf16 = compute_snr(wav, out_bf16)
    diff = snr_bf16 - snr_q8
    faster = "Yes" if time_q8 < time_bf16 else "No"

    comparison_results.append({
        "signal": signal_name,
        "snr_q8_db": round(float(snr_q8), 2),
        "snr_bf16_db": round(float(snr_bf16), 2),
        "snr_diff_db": round(float(diff), 2),
        "time_q8_ms": round(time_q8, 2),
        "time_bf16_ms": round(time_bf16, 2),
    })
    print(f"  {signal_name:<20} {snr_q8:>10.2f} {snr_bf16:>10.2f} "
          f"{diff:>+8.2f} {faster:>12}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("BF16 vs Q8: Mimi Codec Quality Comparison", fontsize=13, fontweight="bold")

names = [r["signal"] for r in comparison_results]
snr_q8s = [r["snr_q8_db"] for r in comparison_results]
snr_bf16s = [r["snr_bf16_db"] for r in comparison_results]
x = np.arange(len(names))
w = 0.35

axes[0].bar(x - w/2, snr_q8s, w, label="Q8 (INT8)", color="#FF9800")
axes[0].bar(x + w/2, snr_bf16s, w, label="BF16 (full)", color="#2196F3")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("SNR (dB)")
axes[0].set_title("Reconstruction Quality (SNR)")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

t_q8s = [r["time_q8_ms"] for r in comparison_results]
t_bf16s = [r["time_bf16_ms"] for r in comparison_results]
axes[1].bar(x - w/2, t_q8s, w, label="Q8 (INT8)", color="#FF9800")
axes[1].bar(x + w/2, t_bf16s, w, label="BF16 (full)", color="#2196F3")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Time (ms)")
axes[1].set_title("Inference Speed")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
plot_path = f"{FOLDERS['comparisons']}/bf16_vs_q8_{ts}.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

json_path = f"{FOLDERS['comparisons']}/bf16_vs_q8_{ts}.json"
with open(json_path, "w") as f:
    json.dump(comparison_results, f, indent=2)

# Free VRAM
del mimi_bf16
torch.cuda.empty_cache()

print(f"\nTest 5 complete. Plot and data saved to Drive.")

In [ ]:
# Test 6: Session report generator
import os, json, glob
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
print("Saving session summary...\n")

def count_files(folder):
    return len([f for f in os.listdir(folder)
                if os.path.isfile(os.path.join(folder, f))])

audio_inputs = count_files(FOLDERS['audio_in'])
audio_outputs = count_files(FOLDERS['audio_out'])
benchmarks = count_files(FOLDERS['benchmarks'])
comparisons = count_files(FOLDERS['comparisons'])

def read_latest_json(folder):
    files = sorted(glob.glob(f"{folder}/*.json"))
    if files:
        with open(files[-1]) as f:
            return json.load(f)
    return None

bench_data = read_latest_json(FOLDERS['benchmarks'])
comp_data = read_latest_json(FOLDERS['comparisons'])

report_lines = [
    "=" * 62,
    " MOSHIKO PROJECT - SESSION REPORT",
    f" Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 62,
    "",
    "1. ENVIRONMENT",
    "-" * 40,
]

env_path = f"{FOLDERS['logs']}/environment_info.json"
if os.path.exists(env_path):
    with open(env_path) as f:
        env = json.load(f)
    report_lines += [
        f"  GPU     : {env.get('gpu_name', 'N/A')}",
        f"  VRAM    : {env.get('vram_total_gb', 'N/A')} GB",
        f"  Model   : {env.get('selected_model', 'N/A')}",
        f"  Python  : {env.get('python_version', 'N/A')}",
    ]

report_lines += [
    "",
    "2. FILES IN DRIVE",
    "-" * 40,
    f"  Audio inputs   : {audio_inputs} file(s)",
    f"  Audio outputs  : {audio_outputs} file(s)",
    f"  Benchmark files: {benchmarks} file(s)",
    f"  Comparison files: {comparisons} file(s)",
]

if bench_data and 'benchmark' in bench_data:
    report_lines += ["", "3. LATEST BENCHMARK RESULTS", "-" * 40]
    report_lines.append(f"  {'Duration':>8} {'Encode(ms)':>12} {'Decode(ms)':>12} {'RTF':>8}")
    for r in bench_data['benchmark']:
        report_lines.append(
            f"  {r['duration_s']:>7}s {r['encode_ms']:>12.1f} "
            f"{r['decode_ms']:>12.1f} {r['rtf']:>8.3f}"
        )

if comp_data:
    report_lines += ["", "4. BF16 vs Q8 COMPARISON", "-" * 40]
    report_lines.append(f"  {'Signal':<20} {'Q8 SNR':>10} {'BF16 SNR':>10} {'Diff':>8}")
    for r in comp_data:
        report_lines.append(
            f"  {r['signal']:<20} {r['snr_q8_db']:>10.2f} "
            f"{r['snr_bf16_db']:>10.2f} {r['snr_diff_db']:>+8.2f}"
        )

report_lines += [
    "",
    "5. HOW TO RESUME NEXT SESSION",
    "-" * 40,
    "  1. Open moshiko2.0.ipynb in Colab",
    "  2. Runtime -> Change runtime type -> T4 GPU",
    "  3. Run all cells in order",
    "  4. Run test.ipynb for tests",
    "",
    "=" * 62,
]

report_text = "\n".join(report_lines)

report_path = f"{FOLDERS['outputs']}/session_report_{ts}.txt"
with open(report_path, 'w') as f:
    f.write(report_text)

print(report_text)
print(f"\nReport saved: {report_path}")

with open(f"{FOLDERS['logs']}/session_log.txt", 'a') as f:
    f.write(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Session completed. Report: session_report_{ts}.txt")

print("\nSession report saved.")

## Troubleshooting

### Common issues

**"CUDA out of memory"**
- Close other notebooks using the GPU
- Reduce MAX_RESPONSE_FRAMES in Test 3
- Restart runtime and re-run moshiko2.0.ipynb

**"File not found" errors**
- Re-run the download cell in moshiko2.0.ipynb
- Check that Drive is mounted

**Test 3 generates no response**
- The LM may need proper text conditioning to generate meaningful output
- Try with real speech audio instead of synthetic tones
- The model is trained for dialogue — it responds to speech, not pure tones

**BF16 comparison fails**
- The BF16 Mimi file may not be downloaded
- Re-run the download cell in moshiko2.0.ipynb

### Next steps

1. **Text-to-Speech**: Requires TTSModel from moshi.models.tts. Needs separate setup with voice conditioning files.
2. **Fine-tuning**: Add a classification layer on top of the LM for specific tasks.
3. **More audio samples**: Upload diverse speech files to test real-world performance.
4. **A/B listening tests**: Have humans rate Q8 vs BF16 audio quality.